In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import optim
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, random_split

from tqdm import tqdm

import torchvision

import torch.nn.functional as F
import torchvision.datasets as datasets
import torchvision.transforms as transforms

!pip install torchmetrics
from torchmetrics import Accuracy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 22.2 MB/s eta 0:00:00


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [160]:
classes = 4

# Define Model
class depthCNN(nn.Module):

    def __init__(self, num_features):
        super(depthCNN, self).__init__()

        self.conv1   = nn.Conv2d(1,   out_channels = 64,  kernel_size = 3, padding =1)
        self.conv2   = nn.Conv2d(64,  out_channels = 64,  kernel_size = 3, padding =1)
        self.conv3   = nn.Conv2d(64,  out_channels = 64,  kernel_size = 3, padding =1)
        self.conv4   = nn.Conv2d(64,  out_channels = 64,  kernel_size = 3, padding =1)
        self.conv5   = nn.Conv2d(64,  out_channels = 64,  kernel_size = 3, padding =1)
        self.pool    = nn.MaxPool2d(kernel_size = 2, stride = 2, padding = 1)
        self.bn1     = nn.BatchNorm2d(64)
        self.bn2     = nn.BatchNorm2d(64)
        self.bn3     = nn.BatchNorm2d(64)
        self.bn4     = nn.BatchNorm2d(64)
        self.bn5     = nn.BatchNorm2d(64)
        self.relu    = nn.ReLU()
        self.softmax = nn.Softmax(dim=-1)
        self.dropout = nn.Dropout(p=0.2)

        self.fullyconnected = nn.LazyLinear(classes)

    def forward(self,x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.pool(x)
        x = self.dropout(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu(x)
        x = self.pool(x)
        x = self.dropout(x)

        x = self.conv3(x)
        x = self.bn3(x)
        x = self.relu(x)
        x = self.pool(x)
        x = self.dropout(x)

        x = self.conv4(x)
        x = self.bn4(x)
        x = self.relu(x)
        x = self.pool(x)
        x = self.dropout(x)

        x = self.conv5(x)
        x = self.bn5(x)
        x = self.relu(x)
        x = self.pool(x)
        x = self.dropout(x)

        x = torch.flatten(x, start_dim = 1)
        x = self.fullyconnected(x)
        x = self.softmax(x)

        return x

In [161]:
# Load training data from .pt file
training_data = torch.load('allData.pt', map_location = device)
waveforms = training_data['raw waves']
labels    = training_data['mag labels']
dataset = TensorDataset(waveforms, labels)

train_perc = 0.7
test_perc  = 0.3

trainingset, testset = random_split(dataset, [train_perc, test_perc])

training = DataLoader(trainingset, batch_size=16)
testing  = DataLoader(testset,     batch_size=16)

In [162]:
# Construct model
model = depthCNN(classes)
model.to(device)

# define optimizer and loss function
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

In [163]:
epochs = 120

# Training
for epoch in range(epochs):

    print(f"Epoch [{epoch + 1}/{epochs}]")

    for i, data in enumerate(training):

        raw_data, labels = data

        labels = torch.squeeze(labels)

        depth_class = model(raw_data)

        loss = criterion(depth_class, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

Epoch [1/120]
Epoch [2/120]
Epoch [3/120]
Epoch [4/120]
Epoch [5/120]
Epoch [6/120]
Epoch [7/120]
Epoch [8/120]
Epoch [9/120]
Epoch [10/120]
Epoch [11/120]
Epoch [12/120]
Epoch [13/120]
Epoch [14/120]
Epoch [15/120]
Epoch [16/120]
Epoch [17/120]
Epoch [18/120]
Epoch [19/120]
Epoch [20/120]
Epoch [21/120]
Epoch [22/120]
Epoch [23/120]
Epoch [24/120]
Epoch [25/120]
Epoch [26/120]
Epoch [27/120]
Epoch [28/120]
Epoch [29/120]
Epoch [30/120]
Epoch [31/120]
Epoch [32/120]
Epoch [33/120]
Epoch [34/120]
Epoch [35/120]
Epoch [36/120]
Epoch [37/120]
Epoch [38/120]
Epoch [39/120]
Epoch [40/120]
Epoch [41/120]
Epoch [42/120]
Epoch [43/120]
Epoch [44/120]
Epoch [45/120]
Epoch [46/120]
Epoch [47/120]
Epoch [48/120]
Epoch [49/120]
Epoch [50/120]
Epoch [51/120]
Epoch [52/120]
Epoch [53/120]
Epoch [54/120]
Epoch [55/120]
Epoch [56/120]
Epoch [57/120]
Epoch [58/120]
Epoch [59/120]
Epoch [60/120]
Epoch [61/120]
Epoch [62/120]
Epoch [63/120]
Epoch [64/120]
Epoch [65/120]
Epoch [66/120]
Epoch [67/120]
Epoc

In [164]:
# Testing
acc = Accuracy(task="multiclass", num_classes = classes).to(device)
model.eval()
with torch.no_grad():
   for i, data in enumerate(testing):

       raw_data, labels = data
       labels = torch.squeeze(labels)
       depth_class = model(raw_data)
       _, preds = torch.max(depth_class, 1)
       _, truth = torch.max(labels, 1)

       acc(preds, truth)

test_accuracy = acc.compute()
print(f"Test accuracy: {test_accuracy}")


Test accuracy: 0.7982119917869568


In [165]:
torch.save(model.state_dict(), 'mag_classifier.pt')